In [ ]:
import zipfile
import pandas as pd
import os
import glob

data_dir = "/Users/tem9735/Desktop/DataProject"
zip_files = sorted(glob.glob(os.path.join(data_dir, "*.zip")))

dfs = []
for zip_path in zip_files:
    quarter = os.path.splitext(os.path.basename(zip_path))[0]
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            if 'sub.txt' not in z.namelist():
                print(f'  [skip] {quarter}: no sub.txt found')
                continue
            with z.open('sub.txt') as f:
                df = pd.read_csv(f, sep='\t', dtype=str, low_memory=False)
                df.insert(0, 'quarter', quarter)
                dfs.append(df)
                print(f'  {quarter}: {len(df):,} rows')
    except Exception as e:
        print(f'  [error] {quarter}: {e}')

print(f'\nMerging {len(dfs)} files...')
merged = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(merged):,}  |  Columns: {len(merged.columns)}')

out_path = os.path.join(data_dir, 'sub_merged.csv')
merged.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

In [ ]:
import pandas as pd
import os

data_dir = "/Users/tem9735/Desktop/DataProject"

# Load merged data
df = pd.read_excel(os.path.join(data_dir, 'sub_merged.xlsx'), dtype=str)

# Companies with >= 2 pure 10-K filings (most recent metadata per CIK)
tenk_counts = df[df['form'] == '10-K'].groupby('cik').size()
qualifying_ciks = tenk_counts[tenk_counts >= 2].index

qualifying = df[df['cik'].isin(qualifying_ciks)].copy()
qualifying = qualifying.sort_values('quarter')
recent = qualifying.drop_duplicates(subset='cik', keep='last')

out_path = os.path.join(data_dir, 'companies_2plus_10k.csv')
recent.to_csv(out_path, index=False)
print(f'Companies with >= 2 10-K filings: {len(recent):,}')
print(f'Saved to {out_path}')

In [5]:
%pip install rapidfuzz

108.13s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for rapidfuzz from https://files.pythonhosted.org/packages/9e/89/c2557e37531d03465193bff0ab9de70b468420a807d71a26a65100635459/rapidfuzz-3.14.5-cp311-cp311-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.8 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path
from rapidfuzz import fuzz, process

# ========= PATHS =========
DATA_DIR  = Path("/Users/tem9735/Desktop/DataProject")
CYBER_DIR = DATA_DIR / "CyberEvents"
OUT_DIR   = DATA_DIR / "FuzzyResults"
OUT_DIR.mkdir(exist_ok=True)

SEC_CSV = DATA_DIR / "companies_2plus_10k.csv"

# Each source file → the column that holds the organization name
SOURCES = {
    "maryland_incidents (1).csv":   "organization",
    "temple_incidents.csv":         "Org Name",
    "veris_export.csv":             "Victim",
    "eurepoc_data.csv":             "receiver_name",
    "ransomware_live_filtered.csv": "Post Title",
}

# ========= CONFIG =========
# auto_accept : score_combo == 100  (perfect match)
# needs_review: score_combo 80–99.99
# low_score   : score_combo < 80

CORP_SUFFIXES = {
    "inc", "inc.", "corp", "corp.", "co", "co.", "company", "companies", "llc",
    "l.l.c.", "ltd", "ltd.", "plc", "plc.", "lp", "l.p.", "llp", "l.l.p.",
    "holdings", "holding", "group", "ag", "s.a.", "s.a", "bv", "b.v.", "p.l.c.",
    "nv", "n.v.", "sa", "gmbh", "limited", "pte", "pte.", "pc", "p.c.",
    "incorporated", "corporation", "partners", "partner", "partnership", "ab", "oyj"
}

NON_TRADED_HINTS = [
    "university", "college", "school", "public schools", "isd", "usd", "district",
    "county", "city of", "town of", "village of", "state of", "department", "dept",
    "police", "sheriff", "fire", "library", "hospital", "medical center",
    "health department", "transportation authority", "water", "utilities",
    "board of", "authority", "academy", "commission", "ministry", "church",
    "diocese", "synagogue", "mosque", "parish", "individual", "clinic"
]

# ========= HELPERS =========
PARENS_RE     = re.compile(r"\([^)]*\)")
NON_WORD_RE   = re.compile(r"[^a-z0-9\s]")
MULTISPACE_RE = re.compile(r"\s+")

def normalize_name(raw):
    if not isinstance(raw, str):
        return ""
    s = unicodedata.normalize("NFKD", raw.strip().lower()).encode("ascii", "ignore").decode("ascii")
    s = s.replace("&", " and ")
    s = PARENS_RE.sub(" ", s)
    s = NON_WORD_RE.sub(" ", s)
    s = MULTISPACE_RE.sub(" ", s).strip()
    tokens = [t for t in s.split() if t and t not in CORP_SUFFIXES]
    return " ".join(sorted(tokens))

def looks_non_traded(raw):
    if not isinstance(raw, str):
        return False
    return any(h in raw.lower() for h in NON_TRADED_HINTS)

def hybrid_scores(a, b):
    ts    = fuzz.token_sort_ratio(a, b, score_cutoff=0)
    pr    = fuzz.partial_ratio(a, b, score_cutoff=0)
    combo = 0.6 * ts + 0.4 * pr
    return ts, pr, combo

def get_decision(combo):
    if combo == 100:
        return "auto_accept"
    elif combo >= 80:
        return "needs_review"
    else:
        return "low_score"

# ========= LOAD SEC =========
sec = pd.read_csv(SEC_CSV, dtype=str)[["cik", "name"]].dropna()
sec["name_norm"] = sec["name"].map(normalize_name)
sec = sec[sec["name_norm"].str.len() > 0]
sec_dedup = sec.groupby("name_norm", as_index=False).first()
sec_norm_list = sec_dedup["name_norm"].tolist()

# ========= MATCH FUNCTION =========
def match_source(file_name, org_col):
    fpath = CYBER_DIR / file_name
    df    = pd.read_csv(fpath, dtype=str) if fpath.suffix.lower() == ".csv" else pd.read_excel(fpath, dtype=str)

    df[org_col]    = df[org_col].astype(str)
    df["org_norm"] = df[org_col].map(normalize_name)
    df["hint_probably_nontraded"] = df[org_col].map(looks_non_traded)

    unique_orgs = df[[org_col, "org_norm", "hint_probably_nontraded"]].drop_duplicates().reset_index(drop=True)

    rows = []
    for _, r in unique_orgs.iterrows():
        org_raw, org_norm, hint = r[org_col], r["org_norm"], bool(r["hint_probably_nontraded"])

        if not org_norm:
            rows.append({"md_organization": org_raw, "md_org_norm": org_norm,
                         "hint_probably_nontraded": hint, "sec_best_name": None,
                         "sec_best_cik": None, "sec_best_norm": None,
                         "score_token_sort": 0, "score_partial": 0,
                         "score_combo": 0, "decision": "no_name_norm"})
            continue

        candidates = process.extract(org_norm, sec_norm_list, scorer=fuzz.token_sort_ratio, limit=1)
        if not candidates:
            rows.append({"md_organization": org_raw, "md_org_norm": org_norm,
                         "hint_probably_nontraded": hint, "sec_best_name": None,
                         "sec_best_cik": None, "sec_best_norm": None,
                         "score_token_sort": 0, "score_partial": 0,
                         "score_combo": 0, "decision": "no_candidates"})
            continue

        sec_norm = candidates[0][0]
        ts, pr, combo = hybrid_scores(org_norm, sec_norm)
        sec_row  = sec_dedup.loc[sec_dedup["name_norm"] == sec_norm].iloc[0]

        rows.append({
            "md_organization":  org_raw,
            "md_org_norm":      org_norm,
            "hint_probably_nontraded": hint,
            "sec_best_name":    sec_row["name"],
            "sec_best_cik":     sec_row["cik"],
            "sec_best_norm":    sec_norm,
            "score_token_sort": round(ts, 3),
            "score_partial":    round(pr, 3),
            "score_combo":      round(combo, 3),
            "decision":         get_decision(combo),
        })

    match_df = pd.DataFrame(rows)
    result   = df.merge(
        match_df.drop_duplicates(subset=["md_organization", "md_org_norm"]),
        left_on=[org_col, "org_norm"],
        right_on=["md_organization", "md_org_norm"],
        how="left"
    )

    stem    = fpath.stem.replace(" ", "_")
    out_csv = OUT_DIR / f"{stem}_fuzzy_matched.csv"
    result.to_csv(out_csv, index=False, quoting=1)

    auto   = (match_df["decision"] == "auto_accept").sum()
    review = (match_df["decision"] == "needs_review").sum()
    low    = (match_df["decision"] == "low_score").sum()
    print(f"[{file_name}]")
    print(f"  Unique orgs : {len(unique_orgs):,}")
    print(f"  auto_accept : {auto:,}  |  needs_review (80-99): {review:,}  |  low_score (<80): {low:,}")
    print(f"  Saved → {out_csv}\n")

# ========= RUN ALL =========
for file_name, org_col in SOURCES.items():
    match_source(file_name, org_col)

print("All done. Results saved to FuzzyResults/")

In [ ]:
import pandas as pd
from pathlib import Path

FUZZY_DIR  = Path("/Users/tem9735/Desktop/DataProject/FuzzyResults")
REVIEW_DIR = Path("/Users/tem9735/Desktop/DataProject/ManualReview")
REVIEW_DIR.mkdir(exist_ok=True)

REVIEW_COLS = [
    "md_organization",
    "md_org_norm",
    "sec_best_name",
    "sec_best_norm",
    "sec_best_cik",
    "score_token_sort",
    "score_partial",
    "score_combo",
    "decision",
    "hint_probably_nontraded",
]

for fpath in sorted(FUZZY_DIR.glob("*_fuzzy_matched.csv")):
    df = pd.read_csv(fpath, dtype=str, engine='python', on_bad_lines='skip')

    cols   = [c for c in REVIEW_COLS if c in df.columns]
    review = (
        df[cols]
        .drop_duplicates(subset=["md_organization", "md_org_norm"])
        .sort_values("score_combo", ascending=False)
        .reset_index(drop=True)
    )

    out = REVIEW_DIR / fpath.name.replace("_fuzzy_matched", "_review")
    review.to_csv(out, index=False)

    auto   = (review["decision"] == "auto_accept").sum()
    manual = (review["decision"] == "needs_review").sum()
    print(f"{fpath.name}")
    print(f"  {len(review):,} unique orgs  |  auto_accept: {auto:,}  |  needs_review: {manual:,}")
    print(f"  → {out}\n")

print("Manual review files saved to ManualReview/")

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR  = Path("/Users/tem9735/Desktop/DataProject")
FUZZY_DIR = DATA_DIR / "FuzzyResults"

# source → (filename, id_col, date_col)  [date_col=None for VERIS: built from Year/Month/Day]
SOURCES = {
    "eurepoc":    ("eurepoc_data_fuzzy_matched.csv",             "name",        "start_date"),
    "maryland":   ("maryland_incidents_(1)_fuzzy_matched.csv",   "slug",        "event_date"),
    "ransomware": ("ransomware_live_filtered_fuzzy_matched.csv", "Post Title",  "Discovered"),
    "temple":     ("temple_incidents_fuzzy_matched.csv",         "Source",      "Date Began"),
    "veris":      ("veris_export_fuzzy_matched.csv",             "Incident ID", None),
}

def parse_veris_date(row):
    def to_int(val):
        try:
            s = str(val).strip()
            return int(float(s)) if s and s not in ("nan", "") else None
        except:
            return None
    y, m, d = to_int(row.get("Year")), to_int(row.get("Month")), to_int(row.get("Day"))
    try:
        if y and m and d:
            return pd.Timestamp(y, m, d)
        elif y and m:
            return pd.Timestamp(y, m, 1)          # month-only → 1st of month
        elif y:
            return pd.Timestamp(y, 1, 1)           # year-only  → Jan 1 (approximate)
    except:
        pass
    return pd.NaT

records = []
for src_name, (fname, id_col, date_col) in SOURCES.items():
    df = pd.read_csv(FUZZY_DIR / fname, dtype=str, engine="python", on_bad_lines="skip")
    hits = df[df["score_combo"] == "100.0"].copy()

    if src_name == "veris":
        hits["_event_date"] = hits.apply(parse_veris_date, axis=1)
    else:
        raw = hits[date_col].str.replace("\u2011", "-", regex=False)  # normalize Unicode hyphens
        s = pd.to_datetime(raw, errors="coerce", utc=True)
        hits["_event_date"] = s.dt.tz_convert(None)

    hits["_event_date"] = pd.to_datetime(hits["_event_date"]).dt.normalize()

    for _, row in hits.iterrows():
        records.append({
            "cik":        row["sec_best_cik"],
            "company":    row["sec_best_name"],
            "event_date": row["_event_date"],
            "source":     src_name,
            "source_id":  row[id_col],
        })

long = pd.DataFrame(records)
long["event_date"] = pd.to_datetime(long["event_date"])

WINDOW = pd.Timedelta(days=7)
rows_wide = []

for cik, grp in long.groupby("cik", sort=False):
    company = grp["company"].iloc[0]
    dated   = grp[grp["event_date"].notna()].sort_values("event_date").reset_index(drop=True)
    undated = grp[grp["event_date"].isna()]

    clusters = []
    for _, row in dated.iterrows():
        placed = False
        for cluster in clusters:
            if abs(row["event_date"] - cluster["anchor"]) <= WINDOW:
                cluster["rows"].append(row)
                cluster["anchor"] = min(cluster["anchor"], row["event_date"])
                placed = True
                break
        if not placed:
            clusters.append({"anchor": row["event_date"], "rows": [row]})

    for cluster in clusters:
        src_ids = {}
        for row in cluster["rows"]:
            src_ids.setdefault(row["source"], []).append(str(row["source_id"]))
        rows_wide.append({
            "cik":           cik,
            "company":       company,
            "event_date":    cluster["anchor"].strftime("%Y-%m-%d"),
            "n_sources":     len(set(r["source"] for r in cluster["rows"])),
            "eurepoc_id":    "; ".join(src_ids["eurepoc"])    if "eurepoc"    in src_ids else None,
            "maryland_id":   "; ".join(src_ids["maryland"])   if "maryland"   in src_ids else None,
            "ransomware_id": "; ".join(src_ids["ransomware"]) if "ransomware" in src_ids else None,
            "temple_id":     "; ".join(src_ids["temple"])     if "temple"     in src_ids else None,
            "veris_id":      "; ".join(src_ids["veris"])      if "veris"      in src_ids else None,
        })

    for _, row in undated.iterrows():
        src = row["source"]
        rows_wide.append({
            "cik": cik, "company": company, "event_date": None, "n_sources": 1,
            "eurepoc_id":    str(row["source_id"]) if src == "eurepoc"    else None,
            "maryland_id":   str(row["source_id"]) if src == "maryland"   else None,
            "ransomware_id": str(row["source_id"]) if src == "ransomware" else None,
            "temple_id":     str(row["source_id"]) if src == "temple"     else None,
            "veris_id":      str(row["source_id"]) if src == "veris"      else None,
        })

wide = pd.DataFrame(rows_wide)
wide = wide.sort_values(["cik", "event_date"]).reset_index(drop=True)
wide.to_csv(DATA_DIR / "matched_score100_wide.csv", index=False)

total   = len(wide)
no_date = wide["event_date"].isna().sum()
multi   = (wide["n_sources"] >= 2).sum()
print(f"Rows: {total}  |  multi-source (n_sources >= 2): {multi}  |  blank date: {no_date}")
print(f"Saved → {DATA_DIR / 'matched_score100_wide.csv'}")
